In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

EVAL_PATH = Path("../data/rag_eval_questions.csv")
#assert EVAL_PATH.exists(), "Create data/rag_eval_questions.csv first (Day 18)."

eval_df = pd.read_csv(EVAL_PATH)
eval_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\rag_eval_questions.csv'

In [3]:
from pathlib import Path
Path("../data").exists(), list(Path("../data").glob("rag_eval*"))

(True, [WindowsPath('../data/rag_eval_questions_template.csv')])

In [ ]:
# Load index

def load_index(index_dir: str):
    chunks = pd.read_csv(Path(index_dir) / "chunks.csv")
    embs = np.load(Path(index_dir) / "embeddings.npy")
    meta_path = Path(index_dir) / "index_metadata.json"
    meta = {}
    if meta_path.exists():
        meta = pd.read_json(meta_path, typ="series").to_dict()
    return chunks, embs, meta

def build_nn(embs: np.ndarray, n_neighbors: int = 50):
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="cosine", algorithm="brute")
    nn.fit(embs)
    return nn

def retrieve(nn, embedder, query: str, k: int = 10):
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    dist, idx = nn.kneighbors(q_emb, n_neighbors=k)
    # cosine similarity = 1 - cosine_distance
    sims = 1.0 - dist[0]
    return idx[0], sims

In [ ]:
# Evaluation
def eval_retrieval(index_dir: str, model_name: str = "all-MiniLM-L6-v2", ks=(1,3,5,10)):
    chunks, embs, meta = load_index(index_dir)
    nn = build_nn(embs, n_neighbors=max(ks))
    embedder = SentenceTransformer(model_name)

    rows = []
    for _, r in eval_df.iterrows():
        q = r["question"]
        expected = int(r["expected_chunk_id"])
        idxs, sims = retrieve(nn, embedder, q, k=max(ks))

        retrieved_chunk_ids = chunks.iloc[idxs]["chunk_id"].astype(int).tolist()

        # rank of expected in top-k
        rank = None
        for i, cid in enumerate(retrieved_chunk_ids, start=1):
            if cid == expected:
                rank = i
                break

        out = {"question": q, "expected_chunk_id": expected, "rank": rank}
        for k in ks:
            out[f"hit@{k}"] = int(rank is not None and rank <= k)
        out["mrr@10"] = 0.0 if (rank is None or rank > 10) else 1.0 / rank
        out["top1_sim"] = float(sims[0]) if len(sims) else None
        rows.append(out)

    detail = pd.DataFrame(rows)
    summary = {"index_dir": index_dir, "model_name": model_name}
    for k in ks:
        summary[f"hit@{k}"] = detail[f"hit@{k}"].mean()
    summary["mrr@10"] = detail["mrr@10"].mean()
    return pd.Series(summary), detail

In [ ]:
# Run evaluation
summary_400, detail_400 = eval_retrieval("../rag", model_name="all-MiniLM-L6-v2", ks=(1,3,5,10))
summary_400, detail_400.head()

In [ ]:
Path("../reports").mkdir(exist_ok=True)

pd.DataFrame([summary_400]).to_csv("../reports/rag_retrieval_summary.csv", index=False)
detail_400.to_csv("../reports/rag_retrieval_detail.csv", index=False)

print("Saved reports/rag_retrieval_summary.csv and rag_retrieval_detail.csv")

In [ ]:
# Ablation study
index_dirs = ["../rag", "../rag_idx_300", "../rag_idx_500"]  # edit to match what you created

summaries = []
for d in index_dirs:
    if Path(d).exists():
        s, _ = eval_retrieval(d, model_name="all-MiniLM-L6-v2", ks=(1,3,5,10))
        summaries.append(s)

ablation = pd.DataFrame(summaries).sort_values("mrr@10", ascending=False)
ablation


In [ ]:
ablation.to_csv("../reports/rag_retrieval_ablations.csv", index=False)